# Phase 4: Quantitative Evaluation, T-NEER Audit & Qualitative Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paviomuniz/deepwork/blob/main/notebooks/02_phase4_evaluation.ipynb)

This notebook executes **Phase 4** using the LoRA adapters saved to Google Drive during Phase 2 & 3:
1. **Task 4.1:** Diacritic-Preserving vs. Diacritic-Stripped WER & CER across Config A, B, and C.
2. **Task 4.2:** Theological Named Entity Error Audit (**T-NEER**) across curated biblical proper nouns and theological terms.
3. **Task 4.3:** Qualitative side-by-side verse comparisons (`COL`, `GAL`, `EPH`) with `IPython.display.Audio` playback.

In [ ]:
# 1. Mount Google Drive, Clone/Pull Repo & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!git clone https://github.com/paviomuniz/deepwork.git || (cd deepwork && git pull)
%cd /content/deepwork
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

In [ ]:
# 2. Locate Saved LoRA Adapters in Google Drive
import os

DRIVE_CKPT_DIR = '/content/drive/MyDrive/deepwork_checkpoints'
LORA_RAW_DIR = f'{DRIVE_CKPT_DIR}/lora_raw/best_adapter'
LORA_SPECAUG_DIR = f'{DRIVE_CKPT_DIR}/lora_specaug/best_adapter'

assert os.path.isdir(LORA_RAW_DIR), f'Missing adapter: {LORA_RAW_DIR}'
assert os.path.isdir(LORA_SPECAUG_DIR), f'Missing adapter: {LORA_SPECAUG_DIR}'
print('Found adapters:')
print(' - Config B (LoRA raw):     ', LORA_RAW_DIR)
print(' - Config C (LoRA+SpecAug): ', LORA_SPECAUG_DIR)

In [ ]:
# 3. Run Phase 4 Evaluation (Tasks 4.1, 4.2, and 4.3)
from src.config import ProjectConfig
from src.phase4_eval import run_phase4_evaluation

config = ProjectConfig()
results = run_phase4_evaluation(
    config=config,
    lora_raw_dir=LORA_RAW_DIR,
    lora_specaug_dir=LORA_SPECAUG_DIR,
    output_dir='outputs',
    batch_size=8,
    num_qualitative=5,
)

In [ ]:
# 4. Qualitative Side-by-Side Verse Comparisons with Audio Playback (Task 4.3)
from IPython.display import Audio, Markdown, display
from src.dataset import load_and_split_bibletts
from src.phase4_eval import filter_raw_split_by_duration

raw_splits = load_and_split_bibletts(config, nt_only=True)
raw_test_filtered = filter_raw_split_by_duration(raw_splits['test'], config)

for sample in results['qualitative_samples']:
    idx = sample['index']
    row = raw_test_filtered[idx]
    md = (
        f"### {sample['verse_id']} (`{sample['book']}`)\n"
        f"- **Theological Entities:** {', '.join(sample['matched_entities']) or 'None'}\n"
        f"- **Ground Truth:** {sample['ground_truth']}\n"
        f"- **Config A (Zero-Shot):** {sample['predictions']['config_a_baseline']}\n"
        f"- **Config B (LoRA Raw):** {sample['predictions']['config_b_lora_raw']}\n"
        f"- **Config C (LoRA + SpecAugment):** {sample['predictions']['config_c_lora_specaug']}"
    )
    display(Markdown(md))
    display(Audio(row['audio']['array'], rate=row['audio']['sampling_rate']))

In [ ]:
# 5. Persist Phase 4 Reports to Google Drive
!mkdir -p /content/drive/MyDrive/deepwork_checkpoints/phase4_outputs
!cp outputs/phase4_results.json /content/drive/MyDrive/deepwork_checkpoints/phase4_outputs/
!cp outputs/phase4_report.md /content/drive/MyDrive/deepwork_checkpoints/phase4_outputs/
print('Saved Phase 4 outputs to Google Drive!')